# Session 10: Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

## 🤝 Breakout Room #1
  - Task 1: Installing Required Libraries
  - Task 2: Set Environment Variables
  - Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  - Task 4: Construct our RAG application
  - Task 5: Evaluating our Application with Ragas
  - Task 6: Making Adjustments and Re-Evaluating
  - ***Activity #1: Implement a Different Reranking Strategy***


## Task 1: Installing Required Libraries

If you have not already done so, install the required libraries using the uv package manager:
``` bash

uv sync

```


## Task 2: Set Environment Variables:

We'll also need to provide our API keys.
> NOTE: In addition to OpenAI's models, this notebook will be using Cohere's Reranker - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

You have two options for supplying your API keys in this session:
- Use environment variables (see Prerequisite #2 in the README.md)
- Provide them via a prompt when the notebook runs

The following code will load all of the environment variables in your `.env`. Then, it checks for the two API keys we need. If they are not there, it will prompt you to provide them.

First, OpenAI's for our LLM/embedding model combination!

Second, Cohere's for our reranking


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

In [3]:
#not needed
import nest_asyncio
nest_asyncio.apply()

from ragas.run_config import RunConfig

run_config = RunConfig(
    timeout=120,       # fail fast instead of hanging forever
    max_retries=2,
    max_wait=10,
    max_workers=1,     # avoid Cursor/Jupyter concurrency pain
)


## Task 3: Synthetic Dataset Generation for Evaluation using Ragas

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using the Health & Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, and stress management.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [4]:

from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()



### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [5]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1", timeout=60, max_retries=2))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_60271/2993170847.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1", timeout=60, max_retries=2))
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_60271/2993170847.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [ ]:
# from ragas.testset import TestsetGenerator

# generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
# dataset = generator.generate_with_langchain_docs(docs, testset_size=10, run_config=run_config)

In [ ]:
# dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,As someone who is always looking for practical...,[The Personal Wellness Guide A Comprehensive R...,The Bird Dog exercise is recommended for lower...,single_hop_specifc_query_synthesizer
1,what i do for lower back pain? exercises help ...,[The Personal Wellness Guide A Comprehensive R...,Lower back pain affects about 80% of adults so...,single_hop_specifc_query_synthesizer
2,How can the Knee-to-Chest Stretch be incorpora...,[The Personal Wellness Guide A Comprehensive R...,The Knee-to-Chest Stretch can be included in a...,single_hop_specifc_query_synthesizer
3,"What is CBT-I, and how can it help manage inso...",[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,"CBT-I, or Cognitive Behavioral Therapy for Ins...",single_hop_specifc_query_synthesizer
4,Wut are sum benifits of chamomile tee for slee...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,"According to the context, herbal teas such as ...",single_hop_specifc_query_synthesizer
5,magnesium help sleep?,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Magnesium supplements (consult healthcare prov...,single_hop_specifc_query_synthesizer
6,what chapter 19 say about work life balance? i...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 19 say maintaining balance between pro...,single_hop_specifc_query_synthesizer
7,Wut are the main stategies for boostin immune ...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 18 recomends several immune-boosting s...,single_hop_specifc_query_synthesizer
8,Wut are sum key tips from PART 5 for bilding h...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 5 recomends starting small (like commitin...,single_hop_specifc_query_synthesizer


## Task 4: Construct our RAG application

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [28]:
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [30]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)


447

### ❓ Question #1:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### Answer:

Chunk_overlap controls how many characters from the end of one chunk get repeated at the start of the next chunk. The idea is to prevent "boundary loss", which is when the main idea or intent of a sentence or definition gets split over two chunks. This is acconplished by having "overlap" of both chunks. Overlap increases retrieval quality  by ensuring each chunk has enough self-contained meaning to match a query. The tradeoff is that overlap increases total tokens stored and searched because of duplicate text, e.g. larger overlap improves recall but costs more and can add unwanted context noise if it’s too large.

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [31]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [32]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [33]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [34]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# 1) re-chunk the docs so each chunk has real guidance text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print("chunks:", len(split_documents))

# 2) rebuild a fresh in-memory Qdrant collection
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_chunks500",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_chunks500",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})



chunks: 45


In [37]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LLM for answering
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context.
Use only the context. If the answer is not in the context, say you don't know.

Question:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def rag_answer(question: str):
    retrieved_docs = retriever.invoke(question)
    context_text = format_docs(retrieved_docs)
    messages = rag_prompt.format_messages(question=question, context=context_text)
    response = llm.invoke(messages)
    return {"response": response.content, "context": retrieved_docs}


In [38]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [39]:
rag_answer("Two ways to improve sleep quality?")

{'response': 'Two ways to improve sleep quality are:\n\n1. Maintain a consistent sleep schedule, even on weekends.\n2. Create a relaxing bedtime routine (such as reading, gentle stretching, or taking a warm bath).',
 'context': [Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': '2fb1b4f9ae76472787de814d56788cf8', '_collection_name': 'use_case_data_chunks500'}, page_content='Chapter 8: Improving Sleep Quality\n\nSleep hygiene refers to habits and practices that promote consistent, quality sleep.\n\nEssential sleep hygiene practices:\n- Maintain a consistent sleep schedule, even on weekends\n- Create a relaxing bedtime routine (reading, gentle stretching, warm bath)\n- Keep your bedroom cool, dark, and quiet\n- Limit screen exposure 1-2 hours before bed\n- Avoid caffeine after 2 PM\n- Exercise regularly, but not too close to bedtime\n- Limit alcohol and heavy meals before bed'),
  Document(metadata={'source': 'data/HealthWellnessGuide.txt', '_id': '54a500bbe7a44b448178d

In [40]:
question = "What are two ways to improve sleep quality?"
out = rag_answer(question)

print("model response:", out["response"])

for i, d in enumerate(out["context"], start=1):
    print("\nDOC", i, "chars:", len(d.page_content))
    print(d.page_content)



model response: Two ways to improve sleep quality are:

1. Maintain a consistent sleep schedule, even on weekends.
2. Create a relaxing bedtime routine, such as reading or gentle stretching.

DOC 1 chars: 498
Chapter 8: Improving Sleep Quality

Sleep hygiene refers to habits and practices that promote consistent, quality sleep.

Essential sleep hygiene practices:
- Maintain a consistent sleep schedule, even on weekends
- Create a relaxing bedtime routine (reading, gentle stretching, warm bath)
- Keep your bedroom cool, dark, and quiet
- Limit screen exposure 1-2 hours before bed
- Avoid caffeine after 2 PM
- Exercise regularly, but not too close to bedtime
- Limit alcohol and heavy meals before bed

DOC 2 chars: 426
Creating an optimal sleep environment:
- Temperature: 65-68 degrees Fahrenheit (18-20 Celsius)
- Darkness: Use blackout curtains or a sleep mask
- Quiet: Consider white noise machines or earplugs
- Comfort: Invest in a quality mattress and pillows

Chapter 9: Understanding 

Now we can produce a node for retrieval!

In [41]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### A - Augmented

Let's create a simple RAG prompt!

In [42]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G - Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [43]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [44]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [45]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [46]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [47]:
response = graph.invoke({"question" : "What exercises help with lower back pain?"})

In [48]:
response["response"]

'Exercises that help with lower back pain include the Cat-Cow Stretch and Bird Dog.'

## Task 5: Evaluating our Application with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [49]:
import pandas as pd
from types import SimpleNamespace

df = pd.read_csv("ragas_testset.csv")

dataset = []
for _, row in df.iterrows():
    sample = SimpleNamespace(
    user_input=row["user_input"],
    response=None,
    retrieved_contexts=None,
)
dataset.append(SimpleNamespace(eval_sample=sample))

In [50]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [52]:
dataset[0].eval_sample.response

'The essential vitamins mentioned in the context that contribute to a balanced diet are Vitamin C, Vitamin D, Vitamin E, and zinc (although technically a mineral). These vitamins are linked to key health functions, such as immunity and overall wellness, and are recommended through the consumption of colorful fruits and vegetables, sunlight, fortified foods, nuts, seeds, and leafy greens.\n\nThey relate to the overall principles of healthy eating outlined in the nutrition section by emphasizing the importance of including a variety of nutrient-dense foods—like fruits, vegetables, nuts, seeds, and fatty fish—to ensure adequate intake of essential vitamins and minerals. This approach supports immune function, nutrient absorption, and overall health, aligning with the goals of effective meal planning and maintaining a balanced diet for wellness.'

In [53]:
# Task 1: Sanity check the testset and run 1 sample end to end

print("type(dataset):", type(dataset))
print("len(dataset):", len(dataset))

row0 = dataset[0]
print("row0 type:", type(row0))
print("has eval_sample:", hasattr(row0, "eval_sample"))

q0 = row0.eval_sample.user_input
out0 = rag_answer(q0)

print("question:", q0)
print("response:", out0["response"])
print("contexts returned:", len(out0["context"]))


type(dataset): <class 'list'>
len(dataset): 1
row0 type: <class 'types.SimpleNamespace'>
has eval_sample: True
question: What are the essential vitamins mentioned in the context that contribute to a balanced diet, and how do they relate to the overall principles of healthy eating outlined in the nutrition section?
response: The essential vitamins mentioned in the context that contribute to a balanced diet are Vitamin C, Vitamin D, and Vitamin E. 

- Vitamin C is abundant in citrus fruits, bell peppers, and strawberries, and it plays a key role in supporting the immune system.
- Vitamin D is obtained through sunlight exposure, fatty fish, and fortified foods, and it is important for bone health and immune function.
- Vitamin E is found in nuts, seeds, and spinach, and it contributes to immune health and antioxidant protection.

These vitamins relate to the overall principles of healthy eating outlined in the nutrition section by emphasizing the importance of consuming a variety of color

In [54]:
# Task 2: Populate responses and retrieved_contexts for the whole dataset

for row in dataset:
    out = rag_answer(row.eval_sample.user_input)
    row.eval_sample.response = out["response"]
    row.eval_sample.retrieved_contexts = [d.page_content for d in out["context"]]

print("done. example response:")
print(dataset[0].eval_sample.response)


done. example response:
The essential vitamins mentioned in the context that contribute to a balanced diet are Vitamin C, Vitamin D, Vitamin E, and zinc (which is a mineral but included in the key nutrients for immunity). These vitamins and minerals are linked to overall health and immune support.

In relation to the overall principles of healthy eating outlined in the nutrition section, these vitamins are obtained through a variety of nutrient-rich foods such as fruits, vegetables, fatty fish, nuts, seeds, and dairy. Consuming a diverse and colorful array of foods ensures an adequate intake of these essential micronutrients, supporting immunity, wellness, and proper bodily functions. This aligns with the emphasis on variety and balanced choices to maintain a healthy diet in meal planning and wellness practices.


Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [55]:
import pandas as pd

samples = dataset.samples if hasattr(dataset, "samples") else dataset

rows = []
for s in samples:
    es = s.eval_sample
    rows.append({
        "user_input": es.user_input,
        "reference": getattr(es, "reference", None),
        "reference_contexts": getattr(es, "reference_contexts", None),
        "response": getattr(es, "response", None),
        "retrieved_contexts": getattr(es, "retrieved_contexts", None),
    })

df = pd.DataFrame(rows)
df.head()


,user_input,reference,reference_contexts,response,retrieved_contexts
0,What are the essential vitamins mentioned in t...,None,None,The essential vitamins mentioned in the contex...,[Micronutrients:\n- Vitamins: Organic compound...


In [ ]:
import pandas as pd
from ragas import EvaluationDataset

samples = dataset.samples if hasattr(dataset, "samples") else dataset

rows = []
for s in samples:
    es = s.eval_sample
    rows.append(
        {
            "user_input": getattr(es, "user_input", None),
            "reference": getattr(es, "reference", None),
            "reference_contexts": getattr(es, "reference_contexts", None),
            "response": getattr(es, "response", None),
            "retrieved_contexts": getattr(es, "retrieved_contexts", None),
        }
    )

df = pd.DataFrame(rows)

# Normalize types so pydantic validation passes
df["user_input"] = df["user_input"].fillna("").astype(str)
df["response"] = df["response"].fillna("").astype(str)

df["retrieved_contexts"] = df["retrieved_contexts"].apply(
    lambda v: v if isinstance(v, list) else ([] if pd.isna(v) else [str(v)])
)

df["reference_contexts"] = df["reference_contexts"].apply(
    lambda v: v if isinstance(v, list) else ([] if pd.isna(v) else [str(v)])
)

# Keep only rows valid for the full metric set you used earlier
# These metrics require: user_input, response, retrieved_contexts, reference, reference_contexts
df_valid = df[
    (df["user_input"].str.len() > 0)
    & (df["response"].str.len() > 0)
    & (df["retrieved_contexts"].apply(lambda x: isinstance(x, list) and len(x) > 0))
    & (df["reference"].notna())
    & (df["reference"].astype(str).str.lower().ne("none"))
    & (df["reference_contexts"].apply(lambda x: isinstance(x, list) and len(x) > 0))
].copy()

print("rows total:", len(df))
print("rows valid for full metric set:", len(df_valid))
print(df_valid[["user_input", "reference", "response"]].head(1))

# Create EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(df_valid)

print("evaluation_dataset features:", evaluation_dataset.features())
print("evaluation_dataset size:", len(evaluation_dataset))



In [57]:
import os
import pandas as pd

# If you know the exact filename, set it here.
CSV_PATH = "ragas_testset.csv"

print("cwd:", os.getcwd())
print("CSV exists:", os.path.exists(CSV_PATH))

df_csv = pd.read_csv(CSV_PATH)
print("df_csv rows:", len(df_csv))
print("df_csv cols:", list(df_csv.columns))
df_csv.head(3)


cwd: /Users/michaeldoran/AIE9/10_Evaluating_RAG_With_Ragas
CSV exists: True
df_csv rows: 31
df_csv cols: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the role of the Psychology Handbook in...,"[""The Mental Health and Psychology Handbook\nA...",The Psychology Handbook serves as a practical ...,Mental Health Advocate,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,Can you explain what Post-Traumatic Stress Dis...,"[""The Mental Health and Psychology Handbook\nA...",Post-Traumatic Stress Disorder (PTSD) can deve...,Mental Health Advocate,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,What are the key components of Mindfulness-Bas...,['PART 2: THERAPEUTIC APPROACHES Chapter 4: Co...,The key components of Mindfulness-Based Stress...,Wellness Coach,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer


In [58]:
import ast
import pandas as pd

# 1) Keep only rows that support the full metric set
df_targets = df_csv.copy()

# Make sure reference_contexts is a list
def to_list(v):
    if isinstance(v, list):
        return v
    if isinstance(v, str):
        v = v.strip()
        if v.startswith("[") and v.endswith("]"):
            try:
                return ast.literal_eval(v)
            except Exception:
                return []
    return []

df_targets["reference_contexts"] = df_targets["reference_contexts"].apply(to_list)

df_targets = df_targets[
    df_targets["reference"].notna()
    & df_targets["reference_contexts"].apply(lambda x: isinstance(x, list) and len(x) > 0)
].copy()

print("rows in df_csv:", len(df_csv))
print("rows valid for full metrics:", len(df_targets))

# 2) Take a small batch to avoid hangs while you validate the flow
df_batch = df_targets.head(5).copy()
print("batch size:", len(df_batch))
display(df_batch[["user_input", "reference"]].head(5))


rows in df_csv: 31
rows valid for full metrics: 31
batch size: 5


,user_input,reference
0,What is the role of the Psychology Handbook in...,The Psychology Handbook serves as a practical ...
1,Can you explain what Post-Traumatic Stress Dis...,Post-Traumatic Stress Disorder (PTSD) can deve...
2,What are the key components of Mindfulness-Bas...,The key components of Mindfulness-Based Stress...
3,What are the key components and benefits of Mi...,"Mindfulness-Based Stress Reduction (MBSR), dev..."
4,What is discussed in Chapter 13 regarding nutr...,Chapter 13 focuses on the emerging field of nu...


We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [ ]:
import pandas as pd

# Use the df_batch you already built (batch size 5). If you named it differently, replace df_batch below.
df_batch = df_targets.head(5).copy()

rows = []
for _, r in df_batch.iterrows():
    q = r["user_input"]

    out = rag_answer(q)
    resp_text = out["response"]
    ctx_docs = out["context"]

    rows.append(
        {
            "user_input": q,
            "reference": r.get("reference"),
            "reference_contexts": [r.get("reference")] if pd.notna(r.get("reference")) else [],
            "response": resp_text,
            "retrieved_contexts": [d.page_content for d in ctx_docs],
        }
    )

df_eval = pd.DataFrame(rows)

print("df_eval shape:", df_eval.shape)
print("null counts:\n", df_eval[["user_input","response","reference","reference_contexts","retrieved_contexts"]].isna().sum())
print("reference_contexts lens:", df_eval["reference_contexts"].apply(len).value_counts().to_dict())
print("retrieved_contexts lens:", df_eval["retrieved_contexts"].apply(len).value_counts().to_dict())
df_eval.head(2)


In [60]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(df_eval)

print("evaluation_dataset size:", len(evaluation_dataset))
print("evaluation_dataset features:", evaluation_dataset.features())


evaluation_dataset size: 5
evaluation_dataset features: ['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference']


In [61]:
from langchain_openai import ChatOpenAI

evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


This is where it breaks 

In [ ]:
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity,
)
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)

baseline_result


/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_55834/2369212619.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_55834/2369212619.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_55834/2369212619.py:1: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import (

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'context_recall': 0.1000, 'faithfulness': 0.9111, 'factual_correctness(mode=f1)': 0.0960, 'answer_relevancy': 0.0000, 'context_entity_recall': 0.0390, 'noise_sensitivity(mode=relevant)': 0.0000}

In [ ]:
import pandas as pd

# evaluation_dataset -> dataframe for quick inspection
df_check = evaluation_dataset.to_pandas()

print("rows:", len(df_check))
print("null counts:\n", df_check[["user_input","response","retrieved_contexts","reference","reference_contexts"]].isna().sum())
print("\nresponse length stats:\n", df_check["response"].astype(str).str.len().describe())

# how many "I don't know" style answers
resp_lower = df_check["response"].astype(str).str.strip().str.lower()
print("\n'i don't know' count:", (resp_lower == "i don't know").sum())
print("'i don't know' contains count:", resp_lower.str.contains("don't know", na=False).sum())

# show 3 sample rows
for i in range(min(3, len(df_check))):
    r = df_check.iloc[i]
    print("\n--- row", i, "---")
    print("q:", r["user_input"][:120])
    print("resp:", str(r["response"])[:200])
    rc = r["retrieved_contexts"]
    print("retrieved_contexts type:", type(rc), "len:", (len(rc) if isinstance(rc, list) else None))
    ref = r["reference"]
    print("has reference:", isinstance(ref, str) and len(ref.strip()) > 0)


In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_51229/3221782877.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))


Next up - we simply evaluate on our desired metrics!

In [ ]:
df_clean = df.dropna(subset=["reference", "reference_contexts"]).reset_index(drop=True)


In [ ]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_pandas(df_clean)

In [ ]:
import os
print("cwd:", os.getcwd())
print("files:", [f for f in os.listdir(".") if "ragas" in f.lower() or "testset" in f.lower()])

cwd: /Users/michaeldoran/AIE9/10_Evaluating_RAG_With_Ragas
files: ['evaluate_ragas.py', 'ragas_testset.csv', 'generate_ragas_testset.py']


In [ ]:
import pandas as pd
df_csv = pd.read_csv("ragas_testset.csv")
print("shape:", df_csv.shape)
print(df_csv.columns.tolist())
print(df_csv[["user_input","reference","reference_contexts"]].isna().sum())
print(df_csv.head(2))

In [ ]:
import json

def to_list(x):
    if isinstance(x, list):
        return x

    if x is None:
        return None

    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return None

    try:
        return json.loads(s)
    except Exception:
        return None

df_csv["reference_contexts"] = df_csv["reference_contexts"].apply(to_list)

print(
    df_csv["reference_contexts"]
    .apply(lambda v: isinstance(v, list))
    .value_counts(dropna=False)
)

reference_contexts
False    19
True     12
Name: count, dtype: int64


In [ ]:
df_csv["response"] = ""
df_csv["retrieved_contexts"] = [[] for _ in range(len(df_csv))]

In [ ]:
for i, row in df_csv.iterrows():
    out = graph.invoke({"question": row["user_input"]})
    df_csv.at[i, "response"] = out.get("response", "")
    ctx = out.get("context", [])
    df_csv.at[i, "retrieved_contexts"] = [c.page_content for c in ctx]


In [ ]:
df_all = df_csv.copy()
df_all["retrieved_contexts"] = df_all["retrieved_contexts"].apply(lambda v: v if isinstance(v, list) else [])
df_all["response"] = df_all["response"].fillna("").astype(str)

In [ ]:
import pandas as pd
import ast

def to_list_str(v):
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x) for x in v if x is not None]
    if isinstance(v, str):
        s = v.strip()
        if s == "":
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if x is not None]
        except Exception:
            pass
        return [s]
    if pd.isna(v):
        return []
    return [str(v)]

df_eval = df_all.copy()

for col in ["user_input", "response", "reference"]:
    if col not in df_eval.columns:
        df_eval[col] = ""
    df_eval[col] = df_eval[col].fillna("").astype(str)

if "reference_contexts" not in df_eval.columns:
    df_eval["reference_contexts"] = [[] for _ in range(len(df_eval))]
df_eval["reference_contexts"] = df_eval["reference_contexts"].apply(to_list_str)

if "retrieved_contexts" not in df_eval.columns:
    df_eval["retrieved_contexts"] = [[] for _ in range(len(df_eval))]
df_eval["retrieved_contexts"] = df_eval["retrieved_contexts"].apply(to_list_str)

for col in ["persona_name", "query_style", "query_length"]:
    if col in df_eval.columns:
        df_eval[col] = df_eval[col].fillna("").astype(str)

required_cols = ["user_input", "response", "retrieved_contexts", "reference", "reference_contexts"]

df_eval = df_eval[
    required_cols
    + [c for c in ["persona_name", "query_style", "query_length"] if c in df_eval.columns]
]

df_eval_valid = df_eval[
    (df_eval["reference"].str.strip() != "")
    & (df_eval["response"].str.strip() != "")
    & (df_eval["reference_contexts"].apply(len) > 0)
    & (df_eval["retrieved_contexts"].apply(len) > 0)
].copy()

print("rows total:", len(df_eval))
print("rows valid for full metric set:", len(df_eval_valid))
print(df_eval_valid.isna().sum())


rows total: 31
rows valid for full metric set: 12
user_input            0
response              0
retrieved_contexts    0
reference             0
reference_contexts    0
persona_name          0
query_style           0
query_length          0
dtype: int64


In [ ]:
import ast
import pandas as pd

df_test = pd.read_csv("ragas_testset.csv")

# Parse list-like column from CSV string into a real Python list
def parse_list(v):
    if isinstance(v, list):
        return v
    if pd.isna(v):
        return []
    try:
        out = ast.literal_eval(v)
        return out if isinstance(out, list) else []
    except Exception:
        return []

df_test["reference_contexts"] = df_test["reference_contexts"].apply(parse_list)

# Add columns we will fill from your RAG graph
df_test["response"] = ""
df_test["retrieved_contexts"] = [[] for _ in range(len(df_test))]

for i, row in df_test.iterrows():
    q = str(row["user_input"])
    result = graph.invoke({"question": q})

    df_test.at[i, "response"] = str(result.get("response", ""))

    ctx_docs = result.get("context", [])
    df_test.at[i, "retrieved_contexts"] = [d.page_content for d in ctx_docs]

# Keep only the fields ragas evaluation expects for your metric set
df_eval_ready = df_test[[
    "user_input",
    "response",
    "retrieved_contexts",
    "reference",
    "reference_contexts",
]].copy()

# Hardening: correct types so pydantic does not fail
df_eval_ready["user_input"] = df_eval_ready["user_input"].fillna("").astype(str)
df_eval_ready["response"] = df_eval_ready["response"].fillna("").astype(str)
df_eval_ready["reference"] = df_eval_ready["reference"].fillna("").astype(str)
df_eval_ready["reference_contexts"] = df_eval_ready["reference_contexts"].apply(lambda v: v if isinstance(v, list) else [])
df_eval_ready["retrieved_contexts"] = df_eval_ready["retrieved_contexts"].apply(lambda v: v if isinstance(v, list) else [])

# Optionally filter rows that cannot support the full metric set
def usable_row(r):
    return (
        len(r["retrieved_contexts"]) > 0
        and len(r["reference_contexts"]) > 0
        and len(r["reference"].strip()) > 0
        and len(r["response"].strip()) > 0
        and len(r["user_input"].strip()) > 0
    )

df_eval_ready = df_eval_ready[df_eval_ready.apply(usable_row, axis=1)].reset_index(drop=True)

print("rows usable for full metric set:", len(df_eval_ready))
df_eval_ready.to_csv("eval_ready.csv", index=False)
print("wrote eval_ready.csv")


rows usable for full metric set: 31
wrote eval_ready.csv


In [ ]:
# Need to use this for the evaluation- had to build a json file to make this work
import json
with open("baseline_result.json", "r", encoding="utf-8") as f:
    baseline_scores = json.load(f)
baseline_scores

{'context_recall': 0.0978494623655914,
 'faithfulness': 0.7779697900665642,
 'factual_correctness(mode=f1)': 0.22354838709677421,
 'answer_relevancy': 0.3342677848298596,
 'context_entity_recall': 0.08303972550172373,
 'noise_sensitivity(mode=relevant)': 0.09946236559139786}

## Task 6: Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!




We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [38]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [39]:
from langchain_cohere import CohereRerank

In [40]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=adjusted_example_retriever,
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context": retrieved_docs}


We can simply rebuild our graph with the new retriever!

In [41]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [42]:
response = adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

'To improve your sleep quality, you can adopt good sleep hygiene practices such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and keeping your bedroom cool, dark, and quiet. Limiting screen exposure 1-2 hours before bed, avoiding caffeine after 2 PM, and exercising regularly (but not too close to bedtime) can also help. Additionally, following a sleep checklist—including setting room temperature to 65-68°F, using blackout curtains or a sleep mask, and ensuring your mattress and pillows are comfortable—can promote better sleep. Incorporating natural remedies like relaxation techniques, herbal teas, and magnesium supplements (after consulting a healthcare provider) may further enhance sleep quality.'

In [43]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.

In [46]:
rerank_dataset[0].eval_sample.response


'The essential vitamins mentioned in the context that contribute to a balanced diet are Vitamin C, Vitamin D, and Vitamin E. \n\nThese vitamins are part of the key nutrients for immunity, which are important for maintaining overall health. They relate to the principles of healthy eating outlined in the nutrition section by being components of a balanced diet that provides the body with necessary nutrients. Consuming a variety of foods such as citrus fruits, bell peppers, strawberries, fatty fish, fortified foods, nuts, seeds, and spinach helps ensure adequate intake of these essential vitamins, supporting proper bodily functions and overall wellness.'

In [49]:
response_0 = rerank_dataset[0].eval_sample.response
contexts_0 = rerank_dataset[0].eval_sample.retrieved_contexts
type(response_0), type(contexts_0), response_0[:200]

(str,
 list,
 'The essential vitamins mentioned in the context that contribute to a balanced diet are Vitamin C, Vitamin D, and Vitamin E. \n\nThese vitamins are part of the key nutrients for immunity, which are impor')

In [ ]:
import pandas as pd

rows = []
for row in rerank_dataset:
    rows.append(
    {
    "user_input": row.eval_sample.user_input,
    "reference": row.eval_sample.reference,
    "reference_contexts": row.eval_sample.reference_contexts,
    "response": row.eval_sample.response,
    "retrieved_contexts": row.eval_sample.retrieved_contexts,
    }
    )

df_rerank = pd.DataFrame(rows)
df_rerank.columns
df_rerank.isna().sum()

In [53]:
import pandas as pd

row0 = rerank_dataset[0].eval_sample
print(sorted(vars(row0).keys()))

['response', 'retrieved_contexts', 'user_input']


In [55]:
import pandas as pd
from ragas import EvaluationDataset

rows = []
for item in rerank_dataset:
    es = item.eval_sample  # this is a SimpleNamespace
    d = vars(es)           # dict of fields on the eval_sample

    rows.append({
        "user_input": d.get("user_input", ""),
        "response": d.get("response", ""),
        "retrieved_contexts": d.get("retrieved_contexts", []),
        "reference": d.get("reference", None),
        "reference_contexts": d.get("reference_contexts", None),
    })

df_rerank = pd.DataFrame(rows)

# Hard requirements for Ragas EvaluationDataset
df_rerank["user_input"] = df_rerank["user_input"].fillna("").astype(str)
df_rerank["response"] = df_rerank["response"].fillna("").astype(str)
df_rerank["retrieved_contexts"] = df_rerank["retrieved_contexts"].apply(lambda v: v if isinstance(v, list) else [])

# Optional fields needed for metrics like factual_correctness and context_recall
df_rerank["reference"] = df_rerank["reference"].where(df_rerank["reference"].notna(), None)
df_rerank["reference_contexts"] = df_rerank["reference_contexts"].apply(
    lambda v: v if isinstance(v, list) else ([] if v is None else [str(v)])
)

rerank_evaluation_dataset = EvaluationDataset.from_pandas(df_rerank)


In [58]:
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity,
)
from ragas import evaluate, RunConfig



/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_46550/163558187.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_46550/163558187.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_46550/163558187.py:1: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import (
/v

In [60]:
from ragas import RunConfig

custom_run_config = RunConfig(timeout=360)


In [ ]:
df_valid = df.copy()

print("df_valid rows:", len(df_valid))
print(df_valid[["user_input", "reference", "reference_contexts"]].isna().sum())


In [ ]:
import ast
import pandas as pd

print("df_csv rows:", len(df_csv))
print("df_rerank rows:", len(df_rerank))
print("df_csv columns:", list(df_csv.columns))
print("df_rerank columns:", list(df_rerank.columns))

# 1) Make sure df_csv has usable reference fields
df_ref = df_csv.copy()

# If reference_contexts is a string like "['...']", convert to list
def to_list(v):
    if isinstance(v, list):
        return v
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, str):
        s = v.strip()
        # try to parse stringified list
        try:
            parsed = ast.literal_eval(s)
            return parsed if isinstance(parsed, list) else [str(parsed)]
        except Exception:
            return [s]
    return [str(v)]

df_ref["reference_contexts"] = df_ref["reference_contexts"].apply(to_list)

# 2) Merge reference fields into df_rerank, but do NOT let pandas create suffix confusion
df_r = df_rerank.copy()

# Drop any existing reference columns so the merged ones keep clean names
for c in ["reference", "reference_contexts"]:
    if c in df_r.columns:
        df_r = df_r.drop(columns=[c])

df_r = df_r.merge(
    df_ref[["user_input", "reference", "reference_contexts"]],
    on="user_input",
    how="left",
)

print("After merge missing counts:")
print(df_r[["reference", "reference_contexts"]].isna().sum())

# 3) If you still have missing reference rows, show how many inputs did not match
missing = df_r["reference"].isna().sum()
print("rows missing reference after merge:", missing)

# 4) Prepare required columns for ragas evaluation
# retrieved_contexts must be a list, response must be a string
if "retrieved_contexts" in df_r.columns:
    df_r["retrieved_contexts"] = df_r["retrieved_contexts"].apply(lambda v: v if isinstance(v, list) else [])
else:
    df_r["retrieved_contexts"] = [[] for _ in range(len(df_r))]

if "response" in df_r.columns:
    df_r["response"] = df_r["response"].fillna("").astype(str)
else:
    df_r["response"] = ""

# Optional columns sometimes show up as NaN and break pydantic. Make them harmless.
for c in ["persona_name", "query_style", "query_length", "synthesizer_name"]:
    if c in df_r.columns:
        df_r[c] = df_r[c].fillna("").astype(str)

# Keep only columns that matter for evaluation (reduces schema issues)
cols_keep = [c for c in ["user_input","response","retrieved_contexts","reference","reference_contexts"] if c in df_r.columns]
df_r_eval = df_r[cols_keep].copy()

print("df_r_eval null counts:")
print(df_r_eval.isna().sum())
print("df_r_eval rows:", len(df_r_eval))



In [ ]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result

In [68]:
from ragas import EvaluationDataset

rerank_evaluation_dataset = EvaluationDataset.from_pandas(df_r_eval)
rerank_evaluation_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference'], len=1)

In [70]:
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity,
)
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)


/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_46550/2163504656.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_46550/2163504656.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/kt/kthn0r7n3tj6bc8j8w5_glkh0000gn/T/ipykernel_46550/2163504656.py:1: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import (

In [71]:
from ragas import EvaluationDataset

rerank_evaluation_dataset = EvaluationDataset.from_pandas(df_r_eval)


In [72]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)

rerank_result


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'context_recall': 0.7500, 'faithfulness': 0.4286, 'factual_correctness(mode=f1)': 0.5000, 'answer_relevancy': 0.9663, 'context_entity_recall': 0.3333, 'noise_sensitivity(mode=relevant)': 0.2857}

### ❓ Question #2:

Which system performed better, on what metrics, and why?

##### Answer:
The reranked list performed better than the baseline list in terms of answer relevancy, context recall, and context entity recall. Reranking surfaces more relevant evidence, which improves context and context entity recall. Reranking intentionally seeks context better aligned the original question (this is its main job), and this strongly improves the quality of answer relevancy.

The baseline list performed better than the reranked list with respect to faithfullness, noise sensitivity, and factual correctness. The decrease in faithfullness and noise sensitivity scores indicate that the reranking pulls in more irrelevant context and contextual noise than the baseline does. The factual correctness metric actually improved, but because I could only get 1 row to generate for my reranking, this metric is meaningless.

### ❓ Question #3:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### Answer:
Benefits and limitations of using synthetic data generation for RAG evaluation-

Benefits:
- Fast coverage and generation of a large variety of questions and question types. These can be generated quickly without waiting on users.

- Cheap & repeatable.
The same size dataset can be regenerated ad-hoc with evaluations rerun after each change and runs can be compared with consistent results.

- Better control of ground truth. 
Synthetic data provides references and reference_contexts on demand, providing context recall and factual correctness even in the absence of labeled data.

- Better safety regulation.
Using synthetic data vastly reduces privacy and compliance risk, especially in health and wellness. This is one of its primary purposes.

Limitations / Pitfalls.
- Unrealistic or inappropriate questions and hallucination behavior.
Synthetic queries often look too clean and too artificial compared to real user questions. They miss messy phrasing, incorrect grammar, lousy spelling, mixed intents, and incomplete context, and poor awareness of idiomatic phrasing.

- Model bias.
If the same or closely similar LLMs generate questions, references, and evaluations, scores can be inflated due to the llm agreeing with its own assumptions and biases.

- Weak ground truth.
References aren't independently verified with synthetic data, so a “reference” can be underspecified or just completely wrong, making any correctness metrics misleading and useless.

- Domain risk masking.
 Synthetic data often underrepresents domain-specific edge cases unless explicitly prompted- this is clearly a big red flag in terms of health and wellness due to safety and liability.

- DIstorted & misleading metrics.
If reference or reference_contexts are missing, related metrics collapse or become meaningless.

The best use for synthetic data is fast iteration and regression testing. Validation can be handled with a smaller set of human written, domain reviewed questions and references, ideally drawn from real users.

### ❓ Question #4:

If you were building a production wellness assistant, which Ragas metrics would be most important to optimize for and why? Consider the healthcare/wellness domain specifically.

##### Answer:
Far & away the most important Ragas metrics to optimize for a production wellness assistant would be Faithfulness, Factual Correctness, LLM Context Recall, and Response Relevancy and here's why:

In terms of Faithfullness, the answer must be grounded in retrieved context for the sake of safety and liability. Hallucinations in the wellness domain have the potential to cause serious harm, to the extent that failure in this regard isn't an option.

Factual Correctness is important for the same reasons. If Factual Correctness isn't optimized, the response can distort facts, mix up dosages, contraindications, timelines, or definitions, which could seriously risk the health and wellbeing of the user. Better retrieval and reranking, better context formatting, stricter answer structures, and evaluation with clinically reviewed references are the best approach here. Prompts should be written to cite and constrain.  Retrieval quality, chunking, and refusal behavior need to be optimized when context alone is insufficient.

LLM Context Recall also matters in the wellness domain because the agent is providing medical wellness guidance. Missing even one critical detail from poor context recall (contraindication, warning sign, interaction, threshold) creates unsafe and potentially harmfull advice. Retriever recall, chunking strategy, k selection, reranking, and query rewriting can all be used to optimize Context Recall. 

Response Relevancy matters because in the health and wellness domain, users are asking for legitimate, actionable guidance. If answers drift, users can lose trust in the guidance and potentially follow incorrect or harmful steps. Optimizing question understanding, intent classification, and answer templates that reflect the asked intent are the best approaches to increasing robustness in this category.



## Activity #1: Implement a Different Reranking Strategy

In this activity, you'll experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different top_n for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

### Activity #1 Findings:

First and foremost, the current version of ragas has a lot of issues when run on a Mac. Yes, I did load that 'cleaner' code from HW9 that was supposed to address some of it, but it didn't help. Almost every single referenece to ragas functions caused catestrophic system hang. Ultimately I couldn't get this to work in the notebook, so I built it using .py scripts keep state explicit and current. 

What I built with GPT's help is a small RAG system in the rag_pipeline.py that loads a guide from the project data folder, chunks it, embeds those chunks via OpenAI embeddings, stores them in an in-memory Qdrant vector store, and answers queries by retrieving the top chunks and prompting the LLM to use only those as context.

Baseline results were evaluated with eval_baseline.py. That script reads ragas_testset.csv, generates a response and retrieved_contexts for each query using the default similarity retriever, saves the outputs to a .csv, then runs RAGAS metrics on a 'valid' subset. Baseline performance was a little weak because retrieval missed reference evidence frequently, so the model produced vague and/or unsupported answers, and the baseline scores reflected that: context_recall 0.50, faithfulness 0.20, factual_correctness 0.08, answer_relevancy 0.1898, context_entity_recall 0.0944, noise_sensitivity 0.00.

Then in eval_mmr.py, I created a retrieval system that uses MMR (Maximal Marginal Relevance) via vector_store.as_retriever, a tuned k, fetch_k, and lambda_mult. Apparently, MMR selects chunks iteratively in order to balance relevance to the query with diversity across all chosen chunks, which reduces redundancy and increases coverage of the evidence. After tuning and regenerating outputs, metrics improved quite a bit: context_recall 0.90, faithfulness 0.82, factual_correctness 0.82, answer_relevancy 0.9553, context_entity_recall 0.3250, noise_sensitivity 0.02.


Here are the comparitive final results: 

Baseline results-
evaluation_dataset size: 5
{'context_recall': 0.5000,
 'faithfulness': 0.2000,
 'factual_correctness(mode=f1)': 0.0800,
 'answer_relevancy': 0.1898,
 'context_entity_recall': 0.0944,
 'noise_sensitivity(mode=relevant)': 0.0000}

MMR results (tuned)-
evaluation_dataset size: 5
{'context_recall': 0.9000,
 'faithfulness': 0.8200,
 'factual_correctness(mode=f1)': 0.8200,
 'answer_relevancy': 0.9553,
 'context_entity_recall': 0.3250,
 'noise_sensitivity(mode=relevant)': 0.0200}
